# KubeLLM v2 — fine-tuning sur le corpus vérifié (corpus_v3)

**Modèle de base** : Qwen2.5-Coder-1.5B-Instruct (Apache 2.0), LoRA en 4 bits avec Unsloth.
**Corpus** : `data/datasets/corpus_v3.jsonl` — chaque exemple a passé le dry-run du serveur Kubernetes et le contrôle de capacité. Les 36 demandes du banc C1 en sont exclues.
**Sortie** : un fichier GGUF (≈ 1 Go) à installer dans Ollama sur le PC, puis mesuré avec `scripts/campaign_yaml.py` contre le même modèle **sans** fine-tuning.

### Avant de commencer
`Exécution → Modifier le type d'exécution → GPU T4`. Puis exécuter les cellules une par une (Maj + Entrée). Durée totale : 30 à 60 min.

In [ ]:
%%capture
# 1. Installation (2 à 4 min)
!pip install -q unsloth
!pip install -q pyyaml

In [ ]:
# 2. Vérification du GPU
import torch
assert torch.cuda.is_available(), "Pas de GPU : Exécution → Modifier le type d'exécution → T4"
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), "Gio")

In [ ]:
# 3. Envoyer le corpus : choisir corpus_v3.jsonl
#    (copié sous Windows avec : cp ~/aikube/data/datasets/corpus_v3.jsonl /mnt/c/Users/Tamft/Downloads/)
from google.colab import files
up = files.upload()
CORPUS = next(iter(up))
print("reçu :", CORPUS)

In [ ]:
# 4. Lecture et séparation entraînement / contrôle (5 %, au moins 10 exemples)
import json, random, collections
rows = [json.loads(l) for l in open(CORPUS, encoding="utf-8") if l.strip()]
random.Random(42).shuffle(rows)
n_eval = max(10, len(rows) // 20)
eval_rows, train_rows = rows[:n_eval], rows[n_eval:]
print(len(rows), "exemples →", len(train_rows), "entraînement,", len(eval_rows), "contrôle")
print(collections.Counter(r["meta"]["category"] for r in rows).most_common())

In [ ]:
# 5. Modèle de base en 4 bits
from unsloth import FastLanguageModel
MAX_LEN = 4096
BASE = "unsloth/Qwen2.5-Coder-1.5B-Instruct"
model, tokenizer = FastLanguageModel.from_pretrained(model_name=BASE, max_seq_length=MAX_LEN,
                                                     load_in_4bit=True, dtype=None)

def to_text(r):
    return tokenizer.apply_chat_template(r["messages"], tokenize=False)

def n_tok(r):
    return len(tokenizer(to_text(r)).input_ids)

lens = [n_tok(r) for r in rows]
print("longueur en jetons : max", max(lens), "· médiane", sorted(lens)[len(lens) // 2])
print("exemples trop longs (écartés) :", sum(l > MAX_LEN for l in lens))
train_rows = [r for r in train_rows if n_tok(r) <= MAX_LEN]
eval_rows = [r for r in eval_rows if n_tok(r) <= MAX_LEN]

In [ ]:
# 6. Adaptateur LoRA
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=32, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=42)
model.print_trainable_parameters()

In [ ]:
# 7. Entraînement : 3 passages sur le corpus, perte calculée UNIQUEMENT sur la réponse (le YAML)
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

train_ds = Dataset.from_list([{"text": to_text(r)} for r in train_rows])
eval_ds = Dataset.from_list([{"text": to_text(r)} for r in eval_rows])
cfg = dict(dataset_text_field="text", per_device_train_batch_size=2, gradient_accumulation_steps=4,
           num_train_epochs=3, learning_rate=2e-4, lr_scheduler_type="cosine", warmup_ratio=0.05,
           logging_steps=5, eval_strategy="epoch", save_strategy="no", optim="adamw_8bit",
           weight_decay=0.01, fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
           seed=42, output_dir="out", report_to="none")
try:
    trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=train_ds,
                         eval_dataset=eval_ds, args=SFTConfig(**cfg))
except TypeError:   # versions récentes de TRL
    trainer = SFTTrainer(model=model, processing_class=tokenizer, train_dataset=train_ds,
                         eval_dataset=eval_ds, args=SFTConfig(**cfg))
trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n",
                                  response_part="<|im_start|>assistant\n")
stats = trainer.train()
print(stats)

In [ ]:
# 8. Courbe d'apprentissage (pour le mémoire)
import matplotlib.pyplot as plt
hist = trainer.state.log_history
tr = [(h["step"], h["loss"]) for h in hist if "loss" in h]
ev = [(h["step"], h["eval_loss"]) for h in hist if "eval_loss" in h]
plt.figure(figsize=(7, 4))
plt.plot(*zip(*tr), label="entraînement")
if ev:
    plt.plot(*zip(*ev), "o-", label="contrôle (5 %)")
plt.xlabel("pas"); plt.ylabel("perte"); plt.legend(); plt.title("KubeLLM v2 — perte")
plt.savefig("kubellm_v2_loss.png", dpi=150, bbox_inches="tight"); plt.show()
json.dump({"base": BASE, "train": len(train_rows), "eval": len(eval_rows), "log_history": hist},
          open("kubellm_v2_training.json", "w"), indent=1)

In [ ]:
# 9. Contrôle rapide : le modèle produit-il du YAML lisible sur les exemples de contrôle ?
import yaml
FastLanguageModel.for_inference(model)
ok = 0
for r in eval_rows[:8]:
    prompt = tokenizer.apply_chat_template(r["messages"][:2], tokenize=False, add_generation_prompt=True)
    ids = tokenizer(prompt, return_tensors="pt").to("cuda")
    out = model.generate(**ids, max_new_tokens=1200, temperature=0.2, do_sample=True)
    txt = tokenizer.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True)
    try:
        docs = [d for d in yaml.safe_load_all(txt) if d]
        good = bool(docs) and all(isinstance(d, dict) and "kind" in d for d in docs)
    except yaml.YAMLError:
        good = False
    ok += good
    print("✅" if good else "❌", r["messages"][1]["content"].split("## Demande de l'utilisateur")[-1].strip()[:80])
print(f"YAML lisible : {ok}/8  (la vraie mesure se fait ensuite avec le serveur Kubernetes, banc C1)")

In [ ]:
# 10. Export GGUF (format d'Ollama), quantification Q4_K_M — 5 à 10 min
import glob, os
model.save_pretrained_gguf("kubellm-v2", tokenizer, quantization_method="q4_k_m")
cands = glob.glob("kubellm-v2*/*.gguf") + glob.glob("*.gguf")
q4 = [f for f in cands if "q4_k_m" in f.lower()]
gguf = sorted(q4 or cands, key=os.path.getsize)[0]   # le Q4_K_M, pas la version 16 bits
os.rename(gguf, "kubellm-v2.Q4_K_M.gguf")
print("fichier : kubellm-v2.Q4_K_M.gguf", round(os.path.getsize("kubellm-v2.Q4_K_M.gguf") / 2**20), "Mio")

In [ ]:
# 11. Téléchargement (le GGUF fait ~1 Go : si le téléchargement échoue, utiliser la cellule 12)
model.save_pretrained("kubellm-v2-lora"); tokenizer.save_pretrained("kubellm-v2-lora")
!zip -qr kubellm-v2-lora.zip kubellm-v2-lora
for f in ["kubellm_v2_training.json", "kubellm_v2_loss.png", "kubellm-v2-lora.zip", "kubellm-v2.Q4_K_M.gguf"]:
    files.download(f)

In [ ]:
# 12. (Option) Copier vers Google Drive au lieu de télécharger
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/aikube
!cp kubellm-v2.Q4_K_M.gguf kubellm_v2_training.json kubellm_v2_loss.png kubellm-v2-lora.zip /content/drive/MyDrive/aikube/
print("copié dans Mon Drive/aikube")

## Ensuite, sur le PC (terminal Ubuntu WSL)
```bash
curl -fsSL https://ollama.com/install.sh | sh          # Ollama, une seule fois
cp /mnt/c/Users/Tamft/Downloads/kubellm-v2.Q4_K_M.gguf ~/aikube/models/kubellm-v2/
cd ~/aikube/models/kubellm-v2 && ollama create kubellm-v2 -f Modelfile
ollama pull qwen2.5-coder:1.5b                          # témoin : même base, sans fine-tuning
ollama list
```